In [2]:

import numpy as np
import tensorflow as tf

SCALE = 256

model = tf.keras.models.load_model("cnn_model.h5")

# Extract weights
conv1_w, conv1_b = model.layers[0].get_weights()
conv2_w, conv2_b = model.layers[1].get_weights()
dense_w, dense_b = model.layers[4].get_weights()

# Float -> Q8.8
conv1_w_q = np.round(conv1_w * SCALE).astype(np.int16)
conv1_b_q = np.round(conv1_b * SCALE).astype(np.int16)

conv2_w_q = np.round(conv2_w * SCALE).astype(np.int16)
conv2_b_q = np.round(conv2_b * SCALE).astype(np.int16)

dense_w_q = np.round(dense_w * SCALE).astype(np.int16)
dense_b_q = np.round(dense_b * SCALE).astype(np.int16)

print("Weights loaded.")
print("Conv1:", conv1_w.shape)
print("Conv2:", conv2_w.shape)
print("Dense:", dense_w.shape)



Weights loaded.
Conv1: (3, 3, 1, 8)
Conv2: (3, 3, 8, 8)
Dense: (1568, 1)


In [3]:

data = np.load("data.npy")

img = data[0]  # (32,32,1) hoặc (32,32)

if len(img.shape) == 2:
    img = np.expand_dims(img, axis=-1)

img_q = np.round(img * SCALE).astype(np.int16)

conv1_out_q = np.zeros((30, 30, 8), dtype=np.int16)

for f in range(8):
    for i in range(30):
        for j in range(30):

            acc = np.int32(0)

            for ki in range(3):
                for kj in range(3):

                    pixel = img_q[i+ki, j+kj, 0]
                    weight = conv1_w_q[ki, kj, 0, f]

                    acc += int(pixel) * int(weight)

            acc = acc >> 8
            acc += int(conv1_b_q[f])

            # ReLU
            if acc < 0:
                acc = 0

            conv1_out_q[i, j, f] = np.int16(acc)

print("Conv1 done.")



Conv1 done.


In [4]:

conv2_out_q = np.zeros((28, 28, 8), dtype=np.int16)

for f in range(8):
    for i in range(28):
        for j in range(28):

            acc = np.int32(0)

            for c in range(8):
                for ki in range(3):
                    for kj in range(3):

                        pixel = conv1_out_q[i+ki, j+kj, c]
                        weight = conv2_w_q[ki, kj, c, f]

                        acc += int(pixel) * int(weight)

            acc = acc >> 8
            acc += int(conv2_b_q[f])

            # ReLU
            if acc < 0:
                acc = 0

            conv2_out_q[i, j, f] = np.int16(acc)

print("Conv2 done.")



Conv2 done.


In [5]:

pool_out_q = np.zeros((14, 14, 8), dtype=np.int16)

for c in range(8):
    for i in range(14):
        for j in range(14):

            block = conv2_out_q[
                i*2:i*2+2,
                j*2:j*2+2,
                c
            ]

            pool_out_q[i, j, c] = np.max(block)

print("MaxPool done.")



MaxPool done.


In [6]:

flat_q = pool_out_q.flatten()

print("Flatten shape:", flat_q.shape)  # phải là 1568

acc = np.int32(0)

for i in range(1568):
    acc += int(flat_q[i]) * int(dense_w_q[i, 0])

acc = acc >> 8
acc += int(dense_b_q[0])

dense_out_q = acc

print("Dense fixed output:", dense_out_q)



Flatten shape: (1568,)
Dense fixed output: -145


In [7]:

# Fixed -> float
fixed_float = dense_out_q / SCALE

# Sigmoid
fixed_pred = 1.0 / (1.0 + np.exp(-fixed_float))

# TensorFlow prediction
img_tf = np.expand_dims(img, axis=0)

tf_pred = model.predict(img_tf, verbose=0)[0][0]

print("TensorFlow output :", tf_pred)
print("Fixed output      :", fixed_pred)
print("Difference        :", abs(tf_pred - fixed_pred))



TensorFlow output : 0.3599225
Fixed output      : 0.3620664767467021
Difference        : 0.0021439782821177644


In [3]:

import cv2
import numpy as np

IMG_SIZE = 32
SCALE = 256

# chọn đúng ảnh muốn test
IMG_PATH = "raw_images/50k/050000_96.png"

# đọc ảnh thật
img = cv2.imread(IMG_PATH, cv2.IMREAD_GRAYSCALE)

# preprocess giống lúc train
img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
img = img / 255.0

# convert Q8.8
img_q = np.round(img * SCALE).astype(np.int16)

# export sang Verilog
with open("image.mem", "w") as f:
    for row in img_q:
        for pixel in row:
            v = int(pixel) & 0xFFFF
            f.write("{:04x}\n".format(v))

print("Image exported to image.mem")



Image exported to image.mem


In [4]:

import numpy as np
import tensorflow as tf

model = tf.keras.models.load_model("cnn_model.h5")

conv1_w, conv1_b = model.layers[0].get_weights()

SCALE = 256

conv1_w_q = np.round(conv1_w * SCALE).astype(np.int16)
conv1_b_q = np.round(conv1_b * SCALE).astype(np.int16)

# Filter 0 only
f0 = conv1_w_q[:, :, 0, 0]

with open("conv1_filter0.mem", "w") as f:
    for val in f0.flatten():
        f.write("{:04x}\n".format(int(val) & 0xFFFF))

with open("conv1_bias0.mem", "w") as f:
    f.write("{:04x}\n".format(int(conv1_b_q[0]) & 0xFFFF))

print("Filter0 exported.")



Filter0 exported.
